# Partie 2 — MVPA : ICA temporelle (variante 2)

**Objectif :** trouver de manière *data-driven* (sans GLM) des cartes spatiales de l'activité cérébrale, puis voir lesquelles correspondent aux tâches story et math de la partie 1.

**Données :** run `tfMRI_LANGUAGE_LR` du HCP (316 volumes, TR = 0.72 s), déjà corrigé du mouvement et recalé en espace MNI 2 mm.

## Étapes

1. **Chargement et vérification** : on charge le fMRI, le T1 et le fichier d'événements, et on vérifie les dimensions et l'espace des images.
2. **Régresseurs de la tâche** : à partir des onsets, on construit les régresseurs *story* et *math* convolués par la HRF (même matrice de design que le GLM de la partie 1).
3. **Préparation des données** : on garde uniquement les voxels du cerveau (masque), on lisse spatialement, on retire les dérives lentes et on standardise chaque voxel. On obtient une matrice *timepoints × voxels*, où **chaque timepoint est un échantillon**.
4. **Choix du nombre de composantes** : une PCA exploratoire (variance expliquée, coude) aide à choisir combien de composantes indépendantes garder.
5. **ICA (FastICA)** : on décompose les données en composantes indépendantes. Chaque composante a un **décours temporel** et une **carte spatiale**.
6. **Cartes spatiales** : on affiche chaque composante retenue séparément, en vue axiale, sur le T1.
7. **Corrélation avec le GLM** : on corrèle les décours des 10 premières composantes (ordre renvoyé par FastICA) avec les régresseurs story et math, puis on relie ces corrélations aux cartes spatiales.

*Remarque : le signe et l'ordre des composantes ICA sont arbitraires ; on interprète donc surtout la valeur absolue des corrélations.*

In [ ]:
import os.path as op
import nibabel as nib
import pandas as pd

# Chemins (temporaires, en attendant une convention de groupe)
DATA_DIR = "../data/miniproject_2026_final"
T1_PATH = op.join(DATA_DIR, "T1/T1w.nii.gz")
FUNC_PATH = op.join(DATA_DIR, "fMRI/tfMRI_LANGUAGE_LR/tfMRI_LANGUAGE_LR.nii.gz")
ONSETS_PATH = op.join(DATA_DIR, "fMRI/tfMRI_LANGUAGE_LR/tfMRI_LANGUAGE_LR_task_onsets.csv")
TR = 0.72

for p in [T1_PATH, FUNC_PATH, ONSETS_PATH]:
    print(op.basename(p), op.exists(p))

img = nib.load(FUNC_PATH)
print("fMRI shape :", img.shape, "| zooms :", img.header.get_zooms())
print(img.affine)

t1 = nib.load(T1_PATH)
print("T1 shape :", t1.shape, "| zooms :", t1.header.get_zooms())

events = pd.read_csv(ONSETS_PATH)
print("events shape:", events.shape, "| columns:", events.columns.tolist())
events.head(15)

### Structure des événements

Les événements du fichier d'onsets sont **hiérarchiques** : un événement englobant contient trois sous-phases successives.

| Niveau englobant | Sous-phases | Correspond à |
|---|---|---|
| `story` (~28 s) | `present_story` → `question_story` → `response_story` | une histoire = un bloc entier (4 au total) |
| `math` (~12 s) | `present_math` → `question_math` → `response_math` | un calcul ; un bloc math en enchaîne plusieurs (9 au total) |

- **present** : écoute de l'histoire ou de l'opération
- **question** : écoute de la question / des deux propositions
- **response** : fenêtre de réponse au bouton
- **cue** : court signal entre deux blocs, annonçant la tâche suivante

Exemple : le premier `story` va de 0.241 s à 28.511 s, soit exactement du début de `present_story` à la fin de `response_story`.

**Conséquence :** on ne met jamais un événement englobant et ses sous-phases dans le même modèle, car on compterait deux fois la même période et les régresseurs seraient colinéaires. Ici, on utilise les **blocs `story` et `math`** pour construire les régresseurs.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from nilearn.glm.first_level import make_first_level_design_matrix

# 1. Inventaire des événements
print(events["trial_type"].value_counts())
print("Durée du run :", img.shape[-1] * TR, "s | dernier événement fini à",
      (events.onset + events.duration).max().round(1), "s")

# 2. Régresseurs de bloc story / math convolués par la HRF
n_vols = img.shape[-1]
frame_times = np.arange(n_vols) * TR
block_events = events[events["trial_type"].isin(["story", "math"])]

dm = make_first_level_design_matrix(
    frame_times, block_events,
    hrf_model="spm",
    drift_model="cosine", high_pass=0.01,
)
print(dm.columns.tolist())

# 3. Visualisation
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(frame_times, dm["story"], label="story")
ax.plot(frame_times, dm["math"], label="math")
ax.set_xlabel("Temps (s)"); ax.legend(); ax.set_title("Régresseurs convolués (HRF)")
plt.show()

print("Corrélation story / math :", np.corrcoef(dm["story"], dm["math"])[0, 1].round(2))

In [ ]:
global_signal = img.get_fdata(dtype=np.float32)[mask_img.get_fdata().astype(bool)].mean(axis=0)
plt.figure(figsize=(12, 2.5))
plt.plot(np.arange(n_vols), global_signal, ".-", ms=3)
plt.xlabel("Volume"); plt.ylabel("Signal moyen"); plt.title("Signal global (brut)")
plt.xlim(0, 40); plt.grid(alpha=0.3); plt.show()

In [ ]:
from nilearn.image import index_img
#retirer les premiers volumes 
N_DROP = 3                                       # volumes de pré-équilibre retirés

img_clean = index_img(img, slice(N_DROP, None))  # (91, 109, 91, 313)
dm_clean = dm.iloc[N_DROP:]                      # mêmes lignes retirées dans la matrice de design
ft_clean = frame_times[N_DROP:]                  # temps d'origine conservés (onsets inchangés)
print("fMRI :", img_clean.shape, "| design :", dm_clean.shape)

In [ ]:
from nilearn.maskers import NiftiMasker
from nilearn.masking import compute_brain_mask
from nilearn.image import new_img_like
from nilearn import plotting

# --- 3a. Masque cerveau ---
# Masque MNI du cerveau, restreint aux voxels où le signal varie réellement
# (les données HCP valent 0 hors du cerveau)
template_mask = compute_brain_mask(img, threshold=0.5).get_fdata().astype(bool)
data_std = img.get_fdata(dtype=np.float32).std(axis=-1)
mask_img = new_img_like(img.slicer[..., 0], template_mask & (data_std > 0))
print("Voxels dans le masque :", int(mask_img.get_fdata().sum()))

plotting.plot_roi(mask_img, bg_img=t1, display_mode="z", cut_coords=6,
                  title="Masque cerveau")
plt.show()

# --- 3b. Masquage + lissage + nettoyage ---
drift_cols = [c for c in dm_clean.columns if c.startswith("drift")]

masker = NiftiMasker(
    mask_img=mask_img,
    smoothing_fwhm=6,
    detrend=True,
    standardize="zscore_sample",
    t_r=TR,
    dtype="float32",
)
X = masker.fit_transform(img_clean, confounds=dm_clean[drift_cols].values)
print("X :", X.shape)      # (313, 211413)

# --- 3c. Vérifications ---
print("NaN :", np.isnan(X).sum(),
      "| moyenne ≈ 0 :", np.allclose(X.mean(axis=0), 0, atol=1e-3),
      "| std ≈ 1 :", np.allclose(X.std(axis=0, ddof=1), 1, atol=1e-2))

In [ ]:
from sklearn.decomposition import PCA

pca = PCA().fit(X)                       # toutes les composantes (≤ 316)
ratio = pca.explained_variance_ratio_
cum = np.cumsum(ratio)

n_show = 50
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(np.arange(1, n_show + 1), ratio[:n_show], "o-", ms=3)
ax[0].set_xlabel("Composante"); ax[0].set_ylabel("Variance expliquée")
ax[0].set_title("Scree plot (50 premières)")
ax[1].plot(np.arange(1, len(cum) + 1), cum, "-")
ax[1].set_xlabel("Nombre de composantes"); ax[1].set_ylabel("Variance cumulée")
ax[1].set_title("Variance cumulée")
for a in ax: a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

for k in [5, 10, 15, 20, 30, 50]:
    print(f"{k:>3} composantes : {cum[k-1]:.1%} de la variance")

### Choix du nombre de composantes

Avant l'ICA, une PCA exploratoire montre combien de dimensions portent l'essentiel de la variance des données.

- **Scree plot :** coude net autour de la 4ᵉ composante (la 1ʳᵉ explique ~9.5 % de la variance, on passe à ~2 % dès la 4ᵉ). Au-delà de 15–20, la courbe est plate : chaque composante apporte < 1 %.
- **Variance cumulée :** 29.0 % avec 10 composantes, 33.6 % avec 15, 36.9 % avec 20. […]

**Choix : 15 composantes**, car ce nombre :
- est au-delà du coude, donc capte la variance structurée ;
- est avant la zone plate, donc évite de modéliser surtout du bruit ;
- respecte la contrainte de l'énoncé (≥ 10 composantes à corréler avec les régresseurs) ;
- évite que plusieurs réseaux fusionnent en une composante (trop peu) ou qu'un réseau se fragmente (trop) ;
- reste lisible : 15 cartes peuvent être affichées séparément.

La stabilité des réseaux principaux sera vérifiée avec 10 et 20 composantes.

In [ ]:
import os
from sklearn.decomposition import FastICA

N_COMP = 15
OUT_DIR = "../outputs/partie2"
os.makedirs(OUT_DIR, exist_ok=True)

# --- 5a. ICA temporelle : échantillons = timepoints, variables = voxels ---
ica = FastICA(n_components=N_COMP, whiten="unit-variance",
              max_iter=1000, random_state=0)
timecourses = ica.fit_transform(X)      # (316, 15)    décours temporels (sources indépendantes)
spatial_maps = ica.mixing_.T            # (15, 211413) carte spatiale de chaque composante
print("Itérations :", ica.n_iter_, "/ 1000")
print("Décours :", timecourses.shape, "| Cartes :", spatial_maps.shape)

# --- 5b. Convention de signe (arbitraire en ICA) ---
# On retourne carte ET décours ensemble pour que le pic principal de chaque carte soit positif
for i in range(N_COMP):
    if spatial_maps[i, np.argmax(np.abs(spatial_maps[i]))] < 0:
        spatial_maps[i] *= -1
        timecourses[:, i] *= -1

# --- 5c. Sauvegarde (pour ne pas relancer l'ICA à chaque fois) ---
np.save(op.join(OUT_DIR, f"ica{N_COMP}_timecourses.npy"), timecourses)
np.save(op.join(OUT_DIR, f"ica{N_COMP}_spatial_maps.npy"), spatial_maps)
print("Sauvegardé dans", OUT_DIR)

In [ ]:
from scipy.stats import zscore
from nilearn.image import index_img

THRESH = 2.5                                   # seuil |z| pour l'affichage
CUTS = [-30, -15, 0, 15, 30, 45, 60]           # coupes axiales (mm, MNI)

# --- 6a. Cartes en z-score, remises dans l'espace cerveau ---
maps_z = zscore(spatial_maps, axis=1)          # chaque carte : moyenne 0, std 1
maps_img = masker.inverse_transform(maps_z)    # image 4D : (91, 109, 91, 15)

# --- 6b. Une figure par composante (sauvegardée pour le rapport) ---
for i in range(N_COMP):
    disp = plotting.plot_stat_map(
        index_img(maps_img, i), bg_img=t1, display_mode="z", cut_coords=CUTS,
        threshold=THRESH, colorbar=True, black_bg=True, title=f"IC {i}",
    )
    disp.savefig(op.join(OUT_DIR, f"ica{N_COMP}_IC{i:02d}_axial.png"), dpi=150)
    plt.show()

# --- 6c. Figure récapitulative (toutes les composantes) ---
fig = plt.figure(figsize=(14, 1.9 * N_COMP))
for i in range(N_COMP):
    ax = fig.add_subplot(N_COMP, 1, i + 1)
    plotting.plot_stat_map(
        index_img(maps_img, i), bg_img=t1, display_mode="z", cut_coords=CUTS,
        threshold=THRESH, colorbar=False, black_bg=True, title=f"IC {i}", axes=ax,
    )
fig.savefig(op.join(OUT_DIR, f"ica{N_COMP}_all_axial.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
from scipy.stats import kurtosis

r_story = np.array([np.corrcoef(timecourses[:, i], dm_clean["story"])[0, 1] for i in range(N_COMP)])
kurt = kurtosis(spatial_maps, axis=1)          # forte kurtosis = carte focalisée (≠ bruit diffus)
pct = (np.abs(maps_z) > THRESH).mean(axis=1) * 100

fig, axes = plt.subplots(N_COMP, 1, figsize=(12, 1.3 * N_COMP), sharex=True)
story_z = zscore(dm_clean["story"].values)
for i, ax in enumerate(axes):
    ax.plot(ft_clean, story_z, color="lightgray", lw=1)
    ax.plot(ft_clean, zscore(timecourses[:, i]), color="C0", lw=1)
    ax.set_ylabel(f"IC {i}", rotation=0, labelpad=20)
    ax.text(1.01, 0.5, f"r={r_story[i]:+.2f}\nkurt={kurt[i]:.0f}", transform=ax.transAxes,
            va="center", fontsize=8)
    ax.set_yticks([])
axes[-1].set_xlabel("Temps (s)  —  gris : régresseur story")
plt.tight_layout(); plt.show()

print(pd.DataFrame({"r_story": r_story.round(2), "kurtosis": kurt.round(1),
                    "% voxels |z|>2.5": pct.round(2)}))

### Résultats de l'ICA (15 composantes)

#### Corrections apportées après un premier essai
Un premier passage montrait deux composantes artefactuelles (chute au début et à la fin du run). Deux corrections ont été faites :
- **retrait des 3 premiers volumes**, dont le signal global est plus élevé (volumes de pré-équilibre) ;
- **retrait des dérives lentes par régression des cosinus** de la matrice de design, à la place du filtre passe-haut (filtre de Butterworth), qui créait des effets de bord.

Après correction, ces composantes ont disparu et la 1ʳᵉ composante PCA est passée de 12.3 % à 9.5 % de variance : une partie de la variance « forte » était donc artefactuelle.

#### Classification des composantes (d'après les décours)
| Type | Composantes | Indice |
|---|---|---|
| **Liées à la tâche** | **IC 11** (r = +0.61 avec story), **IC 6** (r = −0.43, donc active en math) | suivent l'alternance des blocs |
| Mouvement / pics | IC 0, 2, 4, 8 | pics ou chutes brusques et isolés |
| Bruit physiologique | IC 3, 7, 9, 10, 12 | oscillations rapides et régulières (respiration, cardiaque) |
| Focalisée, bruitée | IC 5 | carte très focalisée (kurtosis 6.2), décours bruité : artefact vasculaire ou LCR probable |
| Non liées à la tâche | IC 1, 13, 14 | réseaux de repos possibles ou mélanges |

#### Cartes des composantes liées à la tâche
- **IC 11 (story)** : cortex temporal latéral bilatéral jusqu'au **lobe temporal antérieur** (région ciblée par Binder et al., 2011), cortex préfrontal médian et jonction temporo-pariétale, soit le **réseau sémantique / DMN**. En poids opposé : cortex pariétal et frontal latéral droits.
- **IC 6 (math)** : cortex préfrontal dorsolatéral bilatéral, pré-SMA, **sillon intrapariétal gauche** (traitement des nombres) et cervelet postérieur, soit le **réseau fronto-pariétal / attention dorsale**. En poids opposé : précuneus et cortex cingulaire postérieur (**DMN**, désactivé pendant le calcul).

L'ICA retrouve donc **sans connaître la tâche** les deux réseaux attendus, ainsi que leur opposition story vs math.

#### Points à garder en tête
- **Ordre arbitraire** : FastICA ne trie pas ses composantes (contrairement à la PCA). La composante story (IC 11) se trouve hors des 10 premières demandées par l'énoncé ; l'ordre a d'ailleurs changé entre les deux essais.
- **Signe arbitraire** : seule la valeur absolue des corrélations compte. Comme story et math sont anti-corrélés (r = −0.94), une composante corrélée à l'un l'est presque toujours négativement à l'autre.
- **Cartes peu focalisées** : en ICA *temporelle*, ce sont les décours qui sont rendus indépendants, pas les cartes ; celles-ci ont une faible kurtosis et sont moins nettes qu'en ICA spatiale.
- **Identification visuelle** : les régions sont identifiées à l'œil, à confirmer avec un atlas (Yeo, Harvard-Oxford). Les petites taches en bordure du cerveau ne sont pas interprétées.

In [ ]:
import seaborn as sns

# --- 7a. Régresseurs ---
# Blocs : déjà dans dm_clean
block_regs = dm_clean[["story", "math"]]

# Sous-phases : nouvelle matrice de design (mêmes paramètres HRF, sans dérives)
sub_types = ["present_story", "question_story", "response_story",
             "present_math", "question_math", "response_math", "cue"]
dm_sub = make_first_level_design_matrix(
    frame_times, events[events["trial_type"].isin(sub_types)],
    hrf_model="spm", drift_model=None,
).iloc[N_DROP:][sub_types]                     # mêmes volumes retirés

regs = pd.concat([block_regs, dm_sub], axis=1)  # (313, 9)
print("Régresseurs :", regs.columns.tolist())

# --- 7b. Matrice de corrélation (Pearson) ---
def corr_ic_regs(tc, regs):
    n = tc.shape[1]
    C = np.corrcoef(tc.T, regs.values.T)[:n, n:]
    return pd.DataFrame(C, index=[f"IC {i}" for i in range(n)], columns=regs.columns)

def plot_corr(C, title, fname):
    fig, ax = plt.subplots(figsize=(10, 0.45 * len(C) + 1.5))
    sns.heatmap(C, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1,
                cbar_kws={"label": "r de Pearson"}, ax=ax)
    ax.axvline(2, color="black", lw=2)          # séparation blocs | sous-phases
    ax.set_title(title)
    plt.xticks(rotation=45, ha="right"); plt.tight_layout()
    fig.savefig(op.join(OUT_DIR, fname), dpi=150, bbox_inches="tight")
    plt.show()

# Matrice demandée : les 10 premières composantes (ordre FastICA)
C10 = corr_ic_regs(timecourses[:, :10], regs)
plot_corr(C10, "Corrélation : 10 premières IC × régresseurs du GLM", "ica15_corr_first10.png")

# Version complète (annexe) : les 15 composantes
C15 = corr_ic_regs(timecourses, regs)
plot_corr(C15, "Corrélation : 15 IC × régresseurs du GLM (annexe)", "ica15_corr_all15.png")

# --- 7c. Résumé : composantes les plus liées à la tâche ---
summary = C15.abs().max(axis=1).sort_values(ascending=False)
print("\n|r| max par composante :")
print(pd.DataFrame({"|r| max": summary.round(2),
                    "régresseur": C15.abs().idxmax(axis=1)[summary.index]}).head(6))

### Interprétation des corrélations avec les régresseurs

#### Lecture générale
- La plupart des composantes (IC 0, 2, 3, 5, 7, 8, 9, 10, 12, 13) ont des corrélations faibles (|r| < 0.2) avec tous les régresseurs. C'est cohérent avec leur classification en **mouvement ou bruit physiologique** d'après les décours.
- Quatre composantes sont nettement liées à la tâche. Les **sous-phases** (`present`, `question`, `response`) apportent une information que les blocs seuls ne montrent pas : deux de ces composantes (IC 1 et 4) ne sont visibles qu'avec elles.

#### Composantes liées à la tâche

| IC | Corrélations principales | Régions (carte) | Interprétation |
|---|---|---|---|
| **11** | story **+0.61**, math −0.58 ; present_story +0.42, question_story +0.42, question_math −0.52 | temporal latéral bilatéral jusqu'au lobe temporal antérieur, préfrontal médian, jonction temporo-pariétale | **réseau sémantique / DMN** : actif pendant l'histoire et la question sur son thème |
| **6** | math +0.41, story −0.43 ; **present_math +0.50, present_story −0.61** | préfrontal dorsolatéral bilatéral, pré-SMA, sillon intrapariétal gauche, cervelet postérieur ; opposé : précuneus / cingulaire postérieur | **réseau fronto-pariétal de contrôle** : actif pendant l'écoute des calculs, DMN désactivé |
| **1** | blocs faibles ; **present_math +0.43, question_story −0.45** | pariétal supérieur, précuneus dorsal, sillon intrapariétal, prémoteur ; opposé : temporal postérieur gauche | **attention dorsale / traitement des nombres** vs compréhension de la question (latéralisée à gauche) |
| **4** | blocs ≈ 0 ; **response_math +0.39**, present_math −0.30 | cortex sensorimoteur (zone de la main, surtout à gauche), pariétal inférieur ; opposé : cortex auditif droit | **réponse motrice** (main droite → cortex moteur gauche) vs **écoute auditive** |

L'ICA, sans connaître la tâche, sépare donc non seulement le **sémantique** et le **numérique**, mais aussi différentes **phases** de la tâche : écoute, décision, réponse motrice.

#### Précautions
- **Régresseurs corrélés entre eux** : story et math sont anti-corrélés (r = −0.94) et chaque sous-phase est incluse dans son bloc. Une composante peut donc avoir plusieurs corrélations élevées pour une seule cause.
- **Pas de test de significativité simple** : les décours IRMf sont autocorrélés, donc le nombre de points réellement indépendants est bien inférieur à 313.
- **Corrélations modérées** (max ≈ 0.6) : les décours ICA contiennent aussi du bruit, et la HRF canonique n'est qu'une approximation.
- **IC 4** est peu liée à `response_story` (+0.05) : il n'y a que 4 réponses story contre 9 en math, et l'alternance écoute → réponse est plus marquée en math.
- **Ordre arbitraire** : la composante story (IC 11) se trouve hors des 10 premières demandées par l'énoncé ; elle apparaît dans la matrice complète (15 IC).
- **Identification visuelle** des régions, à confirmer avec un atlas (Yeo, Harvard-Oxford).

In [ ]:
# --- Stabilité : ICA à 10 et 20 composantes ---
ref = {"IC 11 (story)": 11, "IC 6 (math)": 6, "IC 1 (attention)": 1, "IC 4 (réponse)": 4}
rows, matched_maps = [], {}

for n in [10, 20]:
    ica_n = FastICA(n_components=n, whiten="unit-variance", max_iter=2000, random_state=0)
    tc_n = ica_n.fit_transform(X)
    maps_n = ica_n.mixing_.T
    print(f"ICA {n} composantes : {ica_n.n_iter_} itérations")

    # Corrélation spatiale entre les cartes de référence (15) et les nouvelles (n)
    sp = np.corrcoef(spatial_maps, maps_n)[:N_COMP, N_COMP:]

    for name, i in ref.items():
        j = np.argmax(np.abs(sp[i]))            # composante la plus semblable
        s = np.sign(sp[i, j])                    # aligne le signe (arbitraire)
        tc_match = s * tc_n[:, j]
        r_regs = corr_ic_regs(tc_match[:, None], regs).iloc[0]
        rows.append({
            "référence": name, "n_comp": n, "IC trouvée": j,
            "r spatial": abs(sp[i, j]).round(2),
            "r temporel": np.corrcoef(timecourses[:, i], tc_match)[0, 1].round(2),
            "r story": r_regs["story"].round(2),
            "régresseur max": r_regs.abs().idxmax(),
        })
        matched_maps[(name, n)] = s * maps_n[j]

stab = pd.DataFrame(rows)
print(stab.to_string(index=False))
stab.to_csv(op.join(OUT_DIR, "ica_stabilite.csv"), index=False)

# --- Figure : cartes de référence vs correspondances (annexe) ---
fig, axes = plt.subplots(len(ref), 3, figsize=(18, 3.2 * len(ref)),
                         gridspec_kw={"hspace": 0.35, "wspace": 0.05})
for r, (name, i) in enumerate(ref.items()):
    for c, n in enumerate([10, 15, 20]):
        m = spatial_maps[i] if n == 15 else matched_maps[(name, n)]
        plotting.plot_stat_map(
            masker.inverse_transform(zscore(m)), bg_img=t1, display_mode="z",
            cut_coords=[-15, 15, 45], threshold=THRESH, colorbar=False,
            black_bg=True, annotate=False, axes=axes[r, c],
        )
        axes[r, c].set_title(f"{name} — {n} comp.", fontsize=11, color="black")
fig.savefig(op.join(OUT_DIR, "ica_stabilite_cartes.png"), dpi=150, bbox_inches="tight")
plt.show()

### Stabilité des composantes (10, 15 et 20 composantes)

Pour vérifier que le choix de 15 composantes n'influence pas trop les résultats, l'ICA a été relancée avec 10 et 20 composantes (mêmes données, même `random_state`). Pour chaque composante de référence (ICA à 15), on retient la composante dont la **carte spatiale** est la plus semblable (|r| maximal, signe aligné), puis on compare aussi les **décours temporels** et les corrélations avec les régresseurs.

| Composante | 10 composantes (r spatial / temporel) | 20 composantes (r spatial / temporel) |
|---|---|---|
| **Story** (IC 11) | **0.94 / 0.90** : stable | 0.74 / 0.44 : partiellement scindée |
| **Math** (IC 6) | **0.95 / 0.91** : stable | 0.78 / 0.63 : partiellement stable |
| **Attention** (IC 1) | **0.93 / 0.87** : stable | **0.85 / 0.79** : stable |
| **Réponse** (IC 4) | 0.77 / 0.71 : moins spécifique (liée à `question_math`) | **0.83 / 0.75** : stable (`response_math`) |

*Repères : r spatial > 0.7 = stable ; 0.4–0.7 = partiellement stable ; < 0.4 = non retrouvée.*

**Conclusions :**
- Les **quatre composantes liées à la tâche sont retrouvées** dans les trois décompositions : les résultats ne dépendent pas d'un choix précis du nombre de composantes.
- **Avec 10 composantes**, les grands réseaux (story, math, attention) sont quasi identiques, mais la composante réponse se mélange avec la phase de question : **trop peu de composantes fusionnent** des sources distinctes.
- **Avec 20 composantes**, les composantes story et math se **fragmentent** : la carte reste proche, mais une partie du décours passe dans d'autres composantes.
- **15 composantes est donc le meilleur compromis** : les quatre composantes y sont à la fois stables et bien séparées. Cela confirme le choix fait à partir du scree plot.

*Figure en annexe : cartes des quatre composantes pour 10, 15 et 20 composantes (coupes axiales z = −15, 15, 45 mm, seuil |z| > 2.5).*

In [ ]:
from nilearn import datasets, image

# --- Atlas de Schaefer 2018 (400 parcelles, étiquetées selon les 7 réseaux de Yeo) ---
schaefer = datasets.fetch_atlas_schaefer_2018(n_rois=400, yeo_networks=7, resolution_mm=2)

labels = [l.decode() if isinstance(l, bytes) else str(l) for l in schaefer.labels]
if labels[0].lower().startswith("background"):
    labels = labels[1:]                       # selon la version de nilearn

# Chaque parcelle -> son réseau de Yeo (ex. "7Networks_LH_Default_PFC_1" -> Default)
net_keys = ["Vis", "SomMot", "DorsAttn", "SalVentAttn", "Limbic", "Cont", "Default"]
parcel2net = np.zeros(len(labels) + 1, dtype=int)
for p, lab in enumerate(labels, start=1):
    parcel2net[p] = net_keys.index(lab.split("_")[2]) + 1

sch_img = image.resample_to_img(schaefer.maps, mask_img, interpolation="nearest")
mask_bool = mask_img.get_fdata().astype(bool)
parcels = np.round(np.squeeze(sch_img.get_fdata()))[mask_bool].astype(int)
yeo_labels = parcel2net[parcels]              # 0 = hors cortex, 1–7 = réseaux de Yeo
print("Voxels par réseau :", np.bincount(yeo_labels, minlength=8))

net_names = ["Visuel", "Somatomoteur", "Attention dorsale", "Attention ventrale",
             "Limbique", "Fronto-pariétal", "Default mode"]
cols = net_names + ["Hors cortex"]
share = np.array([(yeo_labels == k).mean() for k in range(1, 8)] + [(yeo_labels == 0).mean()]) * 100

# --- Recouvrement des composantes liées à la tâche ---
task_ics = {"IC 11 (story)": 11, "IC 6 (math)": 6, "IC 1 (attention)": 1, "IC 4 (réponse)": 4}
pct, enrich = {}, {}
for name, i in task_ics.items():
    for sign, lab in [(1, "+"), (-1, "−")]:
        sel = sign * maps_z[i] > THRESH
        lbl = yeo_labels[sel]
        p = np.array([(lbl == k).mean() for k in range(1, 8)] + [(lbl == 0).mean()]) * 100
        pct[f"{name} {lab} (n={sel.sum()})"] = p
        enrich[f"{name} {lab}"] = p / share

pct = pd.DataFrame(pct, index=cols).T
enrich = pd.DataFrame(enrich, index=cols).T

fig, ax = plt.subplots(figsize=(11, 5))
sns.heatmap(pct, annot=True, fmt=".0f", cmap="Reds", vmin=0, vmax=60,
            cbar_kws={"label": "% des voxels au-dessus du seuil"}, ax=ax)
ax.set_title("Recouvrement des composantes avec les réseaux de Yeo (|z| > 2.5)")
plt.xticks(rotation=30, ha="right"); plt.tight_layout()
fig.savefig(op.join(OUT_DIR, "ica15_yeo_overlap.png"), dpi=150, bbox_inches="tight")
plt.show()

print("Part de chaque réseau dans le masque (%) :", dict(zip(cols, share.round(1))))
print("\nEnrichissement (> 1 = réseau sur-représenté par rapport à sa taille) :")
print(enrich.round(2))

### Confirmation des réseaux avec l'atlas de Yeo (7 réseaux)

Les 7 réseaux de Yeo et al. (2011) sont obtenus via la parcellisation de Schaefer et al. (2018) (400 parcelles, chacune étiquetée selon Yeo), rééchantillonnée sur la grille 2 mm. Pour chaque composante liée à la tâche, on calcule la répartition de ses voxels au-dessus du seuil (|z| > 2.5) dans les réseaux, séparément pour le côté positif (+) et négatif (−) de la carte, qui correspondent à des conditions opposées. L'**enrichissement** (% observé / part du réseau dans le masque) corrige l'effet de taille des réseaux : > 1 signifie sur-représenté.

| Composante | Côté | Réseaux dominants (enrichissement) | Interprétation |
|---|---|---|---|
| **IC 11 (story)** | + | **Default mode** 43 % (3.0) | réseau sémantique / DMN |
| | − | **Fronto-pariétal** 53 % (6.6), attention dorsale (2.2) | réseau du calcul, en opposition |
| **IC 6 (math)** | + | **Attention dorsale** (4.0), **fronto-pariétal** (3.5), hors cortex 24 % (cervelet) | réseau fronto-pariétal + cervelet |
| | − | **Default mode 85 %** | désactivation du DMN pendant le calcul |
| **IC 1 (attention)** | + | **Attention dorsale** (5.4), fronto-pariétal (3.6) | attention dorsale / traitement des nombres |
| | − | **Default mode** 70 % (4.9) | DMN |
| **IC 4 (réponse)** | + | **Attention ventrale / saillance** (5.5), **somatomoteur** (3.7) | réponse motrice + saillance (décision) |
| | − | **Somatomoteur** (3.5), hors cortex 37 % | cortex auditif + thalamus / noyaux profonds (écoute) |

**Conclusions :**
- L'atlas **confirme les interprétations visuelles** des quatre composantes.
- IC 11 et IC 6 sont presque des **images miroir** : DMN d'un côté, fronto-pariétal / attention dorsale de l'autre. C'est l'opposition classique entre réseau *task-negative* (DMN) et réseaux *task-positive*.

**Nuances :**
- **Cortex auditif** : le réseau « somatomoteur » de Yeo inclut le cortex auditif (gyrus temporal supérieur). Cela explique sa présence dans IC 11 + (parole continue des histoires) et IC 4 − (phase d'écoute).
- **Lobe temporal antérieur absent** : IC 11 n'a presque pas de voxels dans le réseau limbique (1 %), qui contient le lobe temporal antérieur, région sémantique clé selon Binder et al. (2011). Explication probable : **perte de signal par artefacts de susceptibilité** (proximité des sinus et des conduits auditifs).
- **Visuel dans IC 11 +** (25 %) : en partie dû à des taches en bordure occipitale, probablement du bruit ; une contribution de l'imagerie mentale pendant les récits reste possible mais incertaine.
- L'atlas de Yeo ne couvre que le **cortex** : cervelet, thalamus et noyaux profonds sont regroupés dans « hors cortex ».